# Countdown Game

The countdown game is math puzzle where the player is given a set of one or more positive integer tiles, as well as a positive integer target.
The goal is to combine the tiles with an arithmetic operation to produce the target value.
Rules:
- each tile may be used at most once (unused tiles are allowed)
- the only allowed operations are `+`, `-`, `*`, and `/`
- division must be exact, and intermediate values must be positive integers

Example: `tiles = [2, 2, 7, 11]; target = 73; solution = '7*11 - 2 - 2'`

In [1]:
from llm_games.games.countdown import (
    generate_puzzles, reachable_targets, solve, validate_expression
)

## The `countdown` Package

This section goes through some of the important functions from the game package.

`reachable_targets` uses dynamic programming (starting with single tile subsets) to find all possible targets from a list of tiles.
It guarantees finding all possible targets at a minimum operation length, but returns only one valid expression per target.

In [5]:
reachable_targets([2, 5, 5])

{2: Solution(expression='2', value=2, operations=0, depth=0),
 5: Solution(expression='5', value=5, operations=0, depth=0),
 7: Solution(expression='(2 + 5)', value=7, operations=1, depth=1),
 10: Solution(expression='(2 * 5)', value=10, operations=1, depth=1),
 3: Solution(expression='(5 - 2)', value=3, operations=1, depth=1),
 25: Solution(expression='(5 * 5)', value=25, operations=1, depth=1),
 1: Solution(expression='(5 / 5)', value=1, operations=1, depth=1),
 12: Solution(expression='((2 + 5) + 5)', value=12, operations=2, depth=2),
 35: Solution(expression='((2 + 5) * 5)', value=35, operations=2, depth=2),
 15: Solution(expression='((2 * 5) + 5)', value=15, operations=2, depth=2),
 50: Solution(expression='((2 * 5) * 5)', value=50, operations=2, depth=2),
 8: Solution(expression='((5 + 5) - 2)', value=8, operations=2, depth=2),
 20: Solution(expression='(2 * (5 + 5))', value=20, operations=2, depth=2),
 27: Solution(expression='(2 + (5 * 5))', value=27, operations=2, depth=2),
 2

For now, `reachable_targets` supports at most 8 tiles.

In [12]:
try :
    reachable_targets([1, 2, 3, 4, 5, 6, 7, 8, 9])
except Exception as e :
    print(f'{type(e).__name__}: {e}')

ValueError: Exhaustive solver supports at most eight tiles


`solve` simply runs `reachable_targets` then returns the expression associated with the given target (or `None` if that target isn't reachable).

In [13]:
print(
    solve([2, 5, 5], 20)
)
print(
    solve([2, 5, 5], 42)
)

Solution(expression='(2 * (5 + 5))', value=20, operations=2, depth=2)
None


`validate_expression` checks that the expression and puzzle (tiles/target) are legal.
If so, it evaluates the expression and checks whether it matches the target.

In [15]:
expression = '(5 + 5)*2'
numbers = [2, 5, 5]
target = 20
print(
    validate_expression(expression, numbers, target)
)

ValidationResult(legal=True, correct=True, value=20, error=None, used_tiles=(5, 5, 2))


In [19]:
# error contains descriptive error message
print(
    validate_expression('((5 + 5)*2', numbers, target)
)
print(
    validate_expression('4*5', numbers, target)
)
print(
    validate_expression(expression, numbers, 42)
)
print(
    validate_expression('5^2 - 5', numbers, target)
)

ValidationResult(legal=False, correct=False, value=None, error="'(' was never closed (<unknown>, line 1)", used_tiles=())
ValidationResult(legal=False, correct=False, value=None, error='Expression uses unavailable tiles', used_tiles=(4,))
ValidationResult(legal=True, correct=False, value=20, error=None, used_tiles=(5, 5, 2))
ValidationResult(legal=False, correct=False, value=None, error='Only integer tiles and +, -, *, / are allowed', used_tiles=())


`generate_puzzles` generates a list of puzzles following the parameters specified in the kwargs.
Tiles are sampled independently and uniformly, then the target is sampled uniformly from those which fall in the given range.

The `split` kwarg tags puzzles with a string.

The `exclude_keys` kwarg lets you pass earlier puzzle keys to enforce disjoint sets.

In [20]:
puzzles = generate_puzzles(
    count = 10,
    seed = 42,
    split = "example",
    tile_count = 4,
    tile_min= 1,
    tile_max = 20,
    target_min = 1,
    target_max = 200,
    exclude_keys=(),
    max_attempts = None
)
for puzzle in puzzles :
    print(puzzle)

Puzzle(id='example_000000', numbers=(1, 4, 8, 9), target=32, reference_expression='(4 * 8)', min_operations=1, solution_depth=1, split='example')
Puzzle(id='example_000001', numbers=(3, 4, 5, 18), target=84, reference_expression='(4 * (3 + 18))', min_operations=2, solution_depth=2, split='example')
Puzzle(id='example_000002', numbers=(1, 2, 3, 14), target=28, reference_expression='(2 * 14)', min_operations=1, solution_depth=1, split='example')
Puzzle(id='example_000003', numbers=(1, 8, 17, 20), target=177, reference_expression='(17 + (8 * 20))', min_operations=2, solution_depth=2, split='example')
Puzzle(id='example_000004', numbers=(7, 8, 14, 18), target=106, reference_expression='(8 + (7 * 14))', min_operations=2, solution_depth=2, split='example')
Puzzle(id='example_000005', numbers=(1, 6, 9, 19), target=164, reference_expression='((9 * 19) - (1 + 6))', min_operations=3, solution_depth=2, split='example')
Puzzle(id='example_000006', numbers=(5, 9, 11, 14), target=30, reference_expre